# 🏪 Notebook 3: Redis as Cache vs Primary Store

Redis is most commonly used as a **cache**, but it can also serve as a **primary data store** for certain use cases. In this notebook, we'll implement caching patterns, explore TTL and eviction, and build features where Redis is the source of truth — distributed locks, rate limiters, and leaderboards.

## Learning Objectives
- Implement the Cache-Aside pattern with Redis
- Understand TTL, eviction policies, and cache invalidation
- Implement the Write-Through pattern
- Build a distributed lock with Redis
- Build a rate limiter with Redis
- Understand when Redis can be the primary store vs just a cache

## 🛠️ Setup

```bash
cd deep-dives/redis
docker compose up -d
```

### Visualization
- **RedisInsight**: http://localhost:5540 — connect to `redis://localhost:6379`

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right).
If it doesn't appear, reload: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import redis
import json
import time
import random

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Connected to Redis")
    r.flushdb()
    print("🧹 Flushed database for a clean start")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")
    print("   Run: cd deep-dives/redis && docker compose up -d")

# Simulate a "database" using a Python dictionary
# (In a real app, this would be PostgreSQL, MySQL, etc.)
FAKE_DB = {
    "product:1": {"id": 1, "name": "Laptop", "price": 999.99, "stock": 50},
    "product:2": {"id": 2, "name": "Mouse", "price": 29.99, "stock": 200},
    "product:3": {"id": 3, "name": "Keyboard", "price": 79.99, "stock": 150},
    "product:4": {"id": 4, "name": "Monitor", "price": 449.99, "stock": 30},
    "product:5": {"id": 5, "name": "Headphones", "price": 149.99, "stock": 75},
}

def db_read(key):
    """Simulate a slow database read (10ms latency)."""
    time.sleep(0.01)  # Simulated disk I/O latency
    return FAKE_DB.get(key)

def db_write(key, value):
    """Simulate a database write."""
    time.sleep(0.01)
    FAKE_DB[key] = value

print(f"📦 Fake database loaded with {len(FAKE_DB)} products")

## 1️⃣ Cache-Aside Pattern

The most common caching pattern. The application checks the cache first. On a **miss**, it reads from the database and stores the result in the cache.

```
Client → Cache hit?
         ├── YES → Return cached data  ⚡ Fast!
         └── NO  → Read from DB → Store in cache → Return data
```

This is the default pattern for most applications. It's simple, widely understood, and works well.

In [ ]:
def get_product_cache_aside(product_id):
    """Cache-Aside: check cache first, fall back to DB on miss."""
    cache_key = f"product:{product_id}"
    
    # Step 1: Check the cache
    cached = r.get(cache_key)
    if cached:
        print(f"  ⚡ Cache HIT for {cache_key}")
        return json.loads(cached)
    
    # Step 2: Cache miss — read from database
    print(f"  💾 Cache MISS for {cache_key} — reading from DB...")
    data = db_read(cache_key)
    
    if data is None:
        return None
    
    # Step 3: Store in cache with a TTL (time to live)
    r.set(cache_key, json.dumps(data), ex=60)  # Cache for 60 seconds
    print(f"  📝 Stored in cache (TTL=60s)")
    
    return data

# First call: cache miss
print("Request 1:")
product = get_product_cache_aside(1)
print(f"  Result: {product}\n")

# Second call: cache hit
print("Request 2:")
product = get_product_cache_aside(1)
print(f"  Result: {product}\n")

# Different product: cache miss
print("Request 3 (different product):")
product = get_product_cache_aside(2)
print(f"  Result: {product}")

print("\n💡 First request is slow (DB read), subsequent requests are fast (cache hit)!")

In [ ]:
# Let's measure the actual performance difference

# Warm up the cache
for pid in range(1, 6):
    get_product_cache_aside(pid)
print("Cache warmed up!\n")

# Measure cache hits vs misses
def benchmark(label, fetch_fn, product_ids, iterations=50):
    times = []
    for _ in range(iterations):
        pid = random.choice(product_ids)
        start = time.time()
        fetch_fn(pid)
        elapsed = (time.time() - start) * 1000
        times.append(elapsed)
    avg = sum(times) / len(times)
    print(f"{label}: avg={avg:.2f}ms, min={min(times):.2f}ms, max={max(times):.2f}ms")

# All cache hits (data is cached)
benchmark("⚡ Cache hits  ", get_product_cache_aside, [1, 2, 3, 4, 5])

# Force cache misses by flushing
r.flushdb()
benchmark("💾 Cache misses", get_product_cache_aside, [1, 2, 3, 4, 5])

print("\n💡 Cache hits are significantly faster because they skip the database!")

## 2️⃣ TTL and Eviction

**TTL (Time To Live)** automatically expires keys after a set time. This keeps the cache fresh and prevents it from growing unbounded.

**Eviction policies** determine what happens when Redis runs out of memory:

| Policy | Behavior | Best For |
|--------|----------|----------|
| `noeviction` | Returns error when full | Primary data store |
| `allkeys-lru` | Evicts least recently used | General-purpose cache |
| `volatile-lru` | Evicts LRU keys with TTL set | Mixed cache + persistent |
| `allkeys-random` | Evicts random keys | When all keys are equal |
| `volatile-ttl` | Evicts keys closest to expiring | Time-sensitive cache |

In [ ]:
r.flushdb()

# Set a key with a 5-second TTL
r.set("flash_sale:item42", json.dumps({"discount": "50%", "item": "Laptop"}), ex=5)

print("Key set with 5-second TTL")
print(f"  TTL remaining: {r.ttl('flash_sale:item42')} seconds")
print(f"  Value: {r.get('flash_sale:item42')}")

# Wait and check again
time.sleep(2)
print(f"\nAfter 2 seconds:")
print(f"  TTL remaining: {r.ttl('flash_sale:item42')} seconds")
print(f"  Value: {r.get('flash_sale:item42')}")

# Wait for expiration
time.sleep(4)
print(f"\nAfter 6 seconds (past TTL):")
print(f"  TTL: {r.ttl('flash_sale:item42')}")  # -2 means key doesn't exist
print(f"  Value: {r.get('flash_sale:item42')}")  # None

print("\n💡 TTL=-2 means the key has expired and been removed.")
print("   TTL=-1 means the key exists but has no expiration set.")

## 3️⃣ Write-Through Pattern

In Write-Through, every write goes to both the cache AND the database. This ensures the cache is always up to date.

```
Client writes → Update Cache → Update Database → Return OK
                (synchronous — both must succeed)
```

Trade-off: Writes are slower (two writes instead of one), but reads are always fresh.

In [ ]:
r.flushdb()

def write_product_through(product_id, data):
    """Write-Through: update both cache and database."""
    cache_key = f"product:{product_id}"
    
    # Step 1: Write to cache
    r.set(cache_key, json.dumps(data), ex=300)
    print(f"  📝 Written to cache: {cache_key}")
    
    # Step 2: Write to database
    db_write(cache_key, data)
    print(f"  💾 Written to database: {cache_key}")

def read_product_through(product_id):
    """Read from cache (always fresh due to write-through)."""
    cache_key = f"product:{product_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)
    # Fall back to DB if cache was evicted
    data = db_read(cache_key)
    if data:
        r.set(cache_key, json.dumps(data), ex=300)
    return data

# Write a product
print("Writing product:")
write_product_through(1, {"id": 1, "name": "Laptop Pro", "price": 1299.99, "stock": 25})

# Read it back — always hits cache
print("\nReading product:")
product = read_product_through(1)
print(f"  ⚡ Got: {product}")

# Update the price
print("\nUpdating price:")
product["price"] = 1199.99
write_product_through(1, product)

# Read again — cache has the latest value
print("\nReading after update:")
product = read_product_through(1)
print(f"  ⚡ Got: {product}")

print("\n💡 Write-Through keeps cache and DB in sync, but writes are slower.")
print("   Use when read freshness is critical (e.g., inventory counts).")

### Cache-Aside vs Write-Through

| Aspect | Cache-Aside | Write-Through |
|--------|------------|---------------|
| Write path | App writes to DB only | App writes to cache + DB |
| Cache freshness | May serve stale data | Always fresh |
| Write latency | Fast (1 write) | Slower (2 writes) |
| Complexity | Simple | Medium |
| Best for | Read-heavy, tolerates staleness | Read-heavy, needs freshness |

## 4️⃣ Redis as Primary: Distributed Lock

Sometimes Redis IS the database, not just a cache. A distributed lock is a perfect example — the lock state lives only in Redis.

```
Worker 1: INCR lock:ticket_42 → returns 1 → "I got it!" → process → DEL
Worker 2: INCR lock:ticket_42 → returns 2 → "Someone else has it" → wait
```

The TTL ensures the lock is released even if a worker crashes.

In [ ]:
r.flushdb()

class RedisLock:
    """A simple distributed lock using Redis INCR + TTL."""
    
    def __init__(self, redis_client, lock_name, ttl_seconds=10):
        self.r = redis_client
        self.lock_key = f"lock:{lock_name}"
        self.ttl = ttl_seconds
    
    def acquire(self, worker_id):
        """Try to acquire the lock. Returns True if successful."""
        # INCR is atomic — only one worker gets value=1
        count = self.r.incr(self.lock_key)
        
        if count == 1:
            # We're first! Set a TTL so the lock auto-releases on crash
            self.r.expire(self.lock_key, self.ttl)
            print(f"  🔒 {worker_id} acquired the lock")
            return True
        else:
            print(f"  ⏳ {worker_id} failed to acquire — lock is held")
            return False
    
    def release(self, worker_id):
        """Release the lock."""
        self.r.delete(self.lock_key)
        print(f"  🔓 {worker_id} released the lock")

# Simulate two workers trying to book the same ticket
lock = RedisLock(r, "ticket_42", ttl_seconds=10)

print("Scenario: Two workers try to book the same ticket\n")

# Worker 1 acquires the lock
got_lock_1 = lock.acquire("Worker-1")
if got_lock_1:
    print("  Worker-1 is processing the booking...")
    time.sleep(0.1)  # Simulate work

# Worker 2 tries while Worker 1 has the lock
got_lock_2 = lock.acquire("Worker-2")
print(f"  Worker-2 got lock? {got_lock_2}")

# Worker 1 releases
lock.release("Worker-1")

# Now Worker 2 can retry
print()
got_lock_2 = lock.acquire("Worker-2")
if got_lock_2:
    print("  Worker-2 is processing the booking...")
    lock.release("Worker-2")

print("\n💡 INCR is atomic — even with millions of concurrent requests,")
print("   exactly one worker gets value=1 and 'wins' the lock.")

## 5️⃣ Redis as Primary: Rate Limiter

A fixed-window rate limiter using INCR + EXPIRE. The count lives in Redis — it IS the source of truth.

```
Request comes in → INCR rate:user42:window_123 → count ≤ limit? → Allow
                                                  count > limit? → Reject (429)
```

In [ ]:
r.flushdb()

class RateLimiter:
    """Fixed-window rate limiter using Redis INCR + EXPIRE."""
    
    def __init__(self, redis_client, max_requests=5, window_seconds=10):
        self.r = redis_client
        self.max_requests = max_requests
        self.window_seconds = window_seconds
    
    def is_allowed(self, user_id):
        """Check if a request from this user is allowed."""
        # Create a key that includes the current time window
        window = int(time.time()) // self.window_seconds
        key = f"rate:{user_id}:{window}"
        
        # Atomically increment the counter
        count = self.r.incr(key)
        
        # Set TTL on first request so the key auto-expires
        if count == 1:
            self.r.expire(key, self.window_seconds)
        
        allowed = count <= self.max_requests
        remaining = max(0, self.max_requests - count)
        
        return allowed, count, remaining

# Allow 5 requests per 10-second window
limiter = RateLimiter(r, max_requests=5, window_seconds=10)

print("Rate Limiter: 5 requests per 10-second window\n")

# Simulate 8 requests from the same user
for i in range(8):
    allowed, count, remaining = limiter.is_allowed("user_42")
    status = "✅ Allowed" if allowed else "❌ Rate limited (429)"
    print(f"  Request {i+1}: {status} (count={count}, remaining={remaining})")

print("\n💡 Requests 6-8 are rejected. The window resets after 10 seconds.")
print("   INCR + EXPIRE is atomic and handles concurrent requests safely.")

## 6️⃣ Redis as Primary: Leaderboard

Sorted Sets are perfect for leaderboards. The score data lives in Redis — no need for a separate database for rankings.

In [ ]:
r.flushdb()

# Build a real-time gaming leaderboard
def add_score(player, score):
    r.zincrby("leaderboard", score, player)

def get_top_players(count=5):
    return r.zrevrange("leaderboard", 0, count - 1, withscores=True)

def get_player_rank(player):
    rank = r.zrevrank("leaderboard", player)
    score = r.zscore("leaderboard", player)
    return rank + 1 if rank is not None else None, score

# Simulate a game with accumulating scores
players = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hank"]
print("🎮 Simulating 20 game rounds...\n")

for round_num in range(20):
    # Each round, a random player earns points
    player = random.choice(players)
    points = random.randint(10, 100)
    add_score(player, points)

# Display the leaderboard
print("🏆 Final Leaderboard:")
print("=" * 40)
for rank, (player, score) in enumerate(get_top_players(len(players)), 1):
    medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, "  ")
    bar = "█" * int(score / 20)
    print(f"  {medal} #{rank:2d} {player:<10s} {int(score):>4d} pts {bar}")

# Player lookup
print("\nPlayer Lookup:")
for player in ["Alice", "Bob"]:
    rank, score = get_player_rank(player)
    if rank:
        print(f"  {player}: Rank #{rank}, Score: {int(score)}")

total = r.zcard("leaderboard")
print(f"\nTotal players: {total}")
print("\n💡 ZINCRBY, ZREVRANGE, ZREVRANK are all O(log N) — fast even with millions of players!")

## 7️⃣ When to Use Redis as Primary vs Cache

```
Is the data ephemeral (OK to lose)?
├── YES → Redis as primary is fine!
│   Examples: rate limits, sessions, real-time counters, leaderboards
│
└── NO → Redis as cache, durable DB as primary
    Examples: user accounts, orders, payments, inventory

Special cases:
├── Need durability + speed? → Redis with AOF persistence
├── Need strong durability? → AWS MemoryDB (Redis-compatible, disk-backed)
└── Need transactions? → Use PostgreSQL/MySQL as primary
```

| Use Case | Redis Role | Why |
|----------|-----------|-----|
| Page caching | Cache | Data lives in DB, Redis speeds up reads |
| Session storage | Primary | Sessions are ephemeral, speed matters |
| Rate limiting | Primary | Counters are ephemeral, atomicity matters |
| Leaderboards | Primary | Rankings are computable, speed matters |
| Distributed locks | Primary | Lock state is ephemeral by nature |
| User profiles | Cache | Durable data belongs in a database |
| Order history | Cache | Must survive Redis restarts |

## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Cleaned up all keys from this notebook")

## 📚 Summary

### Key Takeaways

1. **Cache-Aside** is the most common pattern — check cache first, fall back to DB on miss
2. **Write-Through** keeps cache always fresh but adds write latency
3. **TTL** prevents stale data and bounds cache size — always set it!
4. **Distributed locks** use INCR + TTL — exactly one worker gets value=1
5. **Rate limiters** use INCR + EXPIRE — simple, atomic, and concurrent-safe
6. **Leaderboards** use Sorted Sets — O(log N) for all operations
7. **Redis as primary** works for ephemeral data (sessions, counters, locks, rankings)
8. **Redis as cache** is better when data must survive restarts (use a durable DB)

### Next Up

In **Notebook 4**, we'll explore **Cluster and Replication** — how to make Redis highly available with Sentinel, scale with clustering, and handle the hot key problem.